# TTM H-DATA-S1 — Colab T4 owner-only one-shot science (H0/H1)

**Operation:** `TTM_HDATA_S1_OWNER_T4_TWO_ARM_ONE_SHOT_128X4096_SCIENCE_DISPATCH_AND_READY_LAST_RECEIPT_NO_FINAL_INFERENCE`.

**One-shot**: 128 AdamW updates per H0/H1 arm, 4096 valid target presentations each update, same frozen P02-1024, BPE and TRAIN/VAL, checkpoint SHA + `READY-last` at 1056/1088/1120/1152. NEW FINAL HF01–HF24, old FINAL/NEXT24, TEST/Q1/Q2/OOD remain sealed.

This notebook does **not** execute science automatically: the dispatch cell is guarded by `ARM_ONE_SHOT = False`. After reviewing the preflight, manually set it to `True` in **the dispatch cell only**. Do not use Run All. For an intentional resume of the *same exact run*, set `RESUME_EXISTING = True`; the scientific runner must refuse any unfinished step after the last durable READY checkpoint (no replay or new run ID).

**Required runtime:** exactly one NVIDIA Tesla T4 and `torch.__version__ == "2.14.1+cu130"`. Never auto-install or change torch; if pin differs, stop (HOLD). No other GPU/CPU substitutes. This notebook must be run by the owner in their Colab session. **GO authorizes one manual dispatch; this notebook is not a result.**

In [ ]:
# 1 — Owner Colab T4 runtime and private Drive (NO MODEL FORWARD)
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from google.colab import drive
drive.mount("/content/drive")
import json, hashlib, pathlib, subprocess, tempfile, sys, zipfile
import torch

def need(cond, code):
    if not cond:
        raise RuntimeError("HDATA_S1_DISPATCH_HOLD:" + code)

def sha_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

need(torch.cuda.is_available(), "NO_CUDA")
need(torch.cuda.device_count() == 1, "NOT_EXACTLY_ONE_GPU")
need("T4" in torch.cuda.get_device_name(0), "GPU_NOT_T4")
need(str(torch.__version__) == "2.14.1+cu130", "TORCH_VERSION_NOT_PINNED")
print("T4_NATIVE_RUNTIME=PASS", torch.cuda.get_device_name(0), torch.__version__)
print("H_DATA_S1_SCIENCE_DISPATCH=NOT_STARTED")

In [ ]:
# 2 — Fetch exact immutable research source (NO SCIENCE EXECUTION)
CHECKOUT_SHA = "7aec0d551c930936412b93eb49951bd1597d71b9"
REPO = pathlib.Path("/content/TTM_HDATA_S1_SCIENCE_FROZEN")
BRANCH = "research/ttm-small4m-h-data-s1-matched-budget-prereg-20261113"
if REPO.exists():
    need((REPO / ".git").exists(), "REPO_PATH_EXISTS_NOT_GIT")
else:
    subprocess.run(["git", "clone", "--quiet", "--single-branch", "--branch", BRANCH,
                    "https://github.com/minhtri22/TokenToModel-Lab.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", CHECKOUT_SHA], cwd=REPO, check=True)
actual_head = subprocess.check_output(["git", "rev-parse", "HEAD"],
                                       cwd=REPO, text=True).strip()
need(actual_head == CHECKOUT_SHA, "FROZEN_COMMIT_CHANGED")
SRC = REPO / "tools/pilot_t4"
HDATA = SRC / "hdata_s1"
RUNNER = HDATA / "ttm_hdata_s1_two_arm_science_runner_candidate_v1.py"
READBACK = HDATA / "ttm_hdata_s1_science_ready_last_readback_v1.py"
QA_SOURCE = HDATA / "ttm_hdata_s1_native_t4_h0_h1_resume_qa_v1.py"
SAMPLER = HDATA / "ttm_hdata_s1_real_metadata_preexec_v1.py"
BASE_RUNNER = SRC / "prompt_conditioning_2x2/ttm_2x2_colab_runner_candidate.py"
P02 = SRC / "integrated/p02_source_exact.py"
LOCK = REPO / "artifacts/qa/ttm-small4m-h-data-s1/preexec_real_train/TTM_HDATA_S1_OWNER_SCIENCE_ONE_SHOT_GO_LOCK_20261113.json"
RIGHTS = REPO / "artifacts/qa/ttm-small4m-h-data-s1/preexec_real_train/TTM_HDATA_S1_NEW_OWNER_RIGHTS_INTERNAL_CDLA_SCOPED_20261113.json"
COMMITMENT = REPO / "docs/research/ttm-small4m-h-data-s1/TTM_HDATA_S1_NEW_FINAL24_PUBLIC_SHA_COMMITMENT_20261113.json"
lock = json.loads(LOCK.read_text("utf-8"))
LOCK_SHA = sha_file(LOCK)

need(lock["authorization"] == "TTM_HDATA_S1_SCIENCE_ONE_SHOT_GO", "GO_LOCK_NOT_GRANTED")
need(lock["release_decision"] == "GO_SCOPED_ONE_SHOT_OWNER_MANUAL_DISPATCH_NO_AUTO_TRAIN",
     "WRONG_RELEASE_DECISION")
for p, key in ((RUNNER, "science_runner_sha256"),
               (QA_SOURCE, "native_resume_qa_source_sha256"),
               (RIGHTS, "new_hdata_rights_receipt_sha256"),
               (COMMITMENT, "public_final24_commitment_sha256")):
    need(sha_file(p) == lock[key], "SHA_DRIFT:" + p.name)
need(lock["science_runner_sha256"] ==
     "eae016e92adc862e71d98920844ebd7c36bacbc3c9157c5a5e33d31aa77ded28",
     "RUNNER_NOT_EXACT_FROZEN_V2")
print("HDATA_S1_FROZEN_SOURCE_AND_GO_LOCK=PASS")
print("SCIENCE_RUNNER_SHA256=" + sha_file(RUNNER))
print("OWNER_GO_LOCK_SHA256=" + LOCK_SHA)
print("SCIENCE_STEPS=0")

In [ ]:
# 3 — Private SHA readback, no FINAL plaintext or model-forward
DRIVE_BASE = pathlib.Path("/content/drive/MyDrive/TTM/ttm-small-pilot")
OWNER_ROOT = DRIVE_BASE / "ttm_small4m_h_data_s1_20261113"
OWNER_PRIVATE = OWNER_ROOT / "owner_private"
RUN_ROOT = OWNER_ROOT / "science_one_shot_v1"
need(OWNER_PRIVATE.is_dir(), "OWNER_PRIVATE_FOLDER_MISSING")
need(str(RUN_ROOT) == lock["science_run_root"], "SCIENCE_RUN_ROOT_NOT_FROZEN")

def find_sha(name, digest):
    paths = sorted(p for p in DRIVE_BASE.rglob(name) if p.is_file() and
                   not p.is_symlink() and sha_file(p) == digest)
    need(bool(paths), "SOURCE_NOT_FOUND_OR_SHA_DRIFT:" + name)
    return paths[0]

PACK = find_sha("T4_PRIVATE_TRAIN_VAL_WINDOWS.npz", lock["train_pack_sha256"])
MANIFEST = find_sha("T4_PRIVATE_PACK_MANIFEST.json",
                    "ef2e6e67d49b1d5a921b1051df329f15574f1c10cece0ce49468acb53322d650")
PARENT = find_sha("checkpoint.pt", lock["checkpoint_parent_sha256"])
BPE = find_sha("T1_BPE2048_REAL_TRAIN_MANIFEST.json", lock["BPE_manifest_sha256"])
QA_RECEIPT = OWNER_PRIVATE / "TTM_HDATA_S1_T4_NATIVE_RESUME_QA_RECEIPT_20261113.json"
need(sha_file(QA_RECEIPT) == lock["native_resume_qa_receipt_sha256"],
     "OWNER_T4_QA_RECEIPT_SHA_DRIFT")
qa = json.loads(QA_RECEIPT.read_text("utf-8"))
need(qa["status"] == "PASS_NATIVE_T4_FULL_INTERRUPTED_RESUME_EXACT_H0_H1" and
     qa["science_optimizer_updates"] == 0 and qa["qa_optimizer_updates"] == 8,
     "T4_QA_NO_SCIENCE_SCOPE_CHANGED")
need({x["arm"] for x in qa["arms"]} == {"H0", "H1"} and
     all(x["continuous_vs_interrupted"] == "PASS_FULL_STATE_EXACT" for x in qa["arms"]),
     "T4_QA_ARM_PARITY_MISSING")

AES_ZIP = OWNER_PRIVATE / "TTM_HDATA_S1_OWNER_PRIVATE_ESCROW_BACKUP_DO_NOT_GITHUB.zip"
need(sha_file(AES_ZIP) ==
     "c5c5d6eae6ce38d892348878ae200d5bbcd42406b53f7b0c43e8876a145d48d4",
     "OWNER_AES_BACKUP_SHA_DRIFT")
ESCROW_NAME = "TTM_HDATA_S1_ENCRYPTED_BLIND_ESCROW_PUBLIC_SAFE.json"
with zipfile.ZipFile(AES_ZIP) as z:
    need(ESCROW_NAME in z.namelist(), "ESCROW_CIPHERTEXT_MISSING")
    # Only encrypted ciphertext, deliberately do NOT read AES key or FINAL text.
    escrow_bytes = z.read(ESCROW_NAME)
need(hashlib.sha256(escrow_bytes).hexdigest() ==
     lock["fresh_final24_escrow_sha256"], "SEALED_ESCROW_SHA_DRIFT")
print("H_DATA_S1_PRIVATE_PINS_AND_AES_CIPHERTEXT=PASS")
print("NEW_FINAL=SEALED")
print("READY_ROOT_EXISTS=" + str(RUN_ROOT.exists()))
print("SCIENCE_TRAINING=NOT_STARTED_BY_PREFLIGHT")

In [ ]:
# 4 — MANUAL OWNER SCIENCE DISPATCH: intentionally fail-closed by default.
# After reading the PASS preflight, set ARM_ONE_SHOT=True and run THIS cell once.
# Set RESUME_EXISTING=True ONLY to continue the SAME run at an intact READY-last.
ARM_ONE_SHOT = False
RESUME_EXISTING = False

if not ARM_ONE_SHOT:
    print("HDATA_S1_OWNER_MANUAL_DISPATCH=NOT_REQUESTED")
    print("Set ARM_ONE_SHOT=True in this cell to start the one frozen H0/H1 science run.")
else:
    need(sha_file(LOCK) == LOCK_SHA, "GO_LOCK_CHANGED_SINCE_PREFLIGHT")
    need(sha_file(RUNNER) == lock["science_runner_sha256"],
         "SCIENCE_RUNNER_CHANGED_SINCE_PREFLIGHT")
    if RESUME_EXISTING:
        need(RUN_ROOT.is_dir(), "RESUME_ROOT_NOT_EXIST")
    else:
        need(not RUN_ROOT.exists(), "ONE_SHOT_RUN_ALREADY_EXISTS_NO_RESTART")
    with tempfile.TemporaryDirectory(prefix="hdata_s1_ciphertext_only_") as tmp:
        escrow = pathlib.Path(tmp) / ESCROW_NAME
        escrow.write_bytes(escrow_bytes)
        need(sha_file(escrow) == lock["fresh_final24_escrow_sha256"],
             "TEMP_ENCRYPTED_ESCROW_DRIFT")
        cmd = [
            sys.executable, str(RUNNER), "--mode", "science",
            "--lock", str(LOCK), "--lock-sha", LOCK_SHA,
            "--qa-receipt", str(QA_RECEIPT),
            "--rights-receipt", str(RIGHTS),
            "--escrow", str(escrow), "--escrow-sha", sha_file(escrow),
            "--runner-qa", str(QA_SOURCE),
            "--sampler", str(SAMPLER),
            "--base", str(BASE_RUNNER),
            "--p02", str(P02),
            "--pack", str(PACK),
            "--manifest", str(MANIFEST),
            "--parent", str(PARENT),
            "--bpe", str(BPE),
            "--commitment", str(COMMITMENT),
            "--root", str(RUN_ROOT),
        ]
        if RESUME_EXISTING:
            cmd.append("--resume")
        print("HDATA_S1_OWNER_T4_ONE_SHOT_DISPATCH=EXECUTING_LOCKED_SCIENCE", flush=True)
        subprocess.run(cmd, cwd=str(REPO), check=True)
        print("HDATA_S1_OWNER_T4_ONE_SHOT=COMPLETED_REQUIRES_READY_READBACK", flush=True)
# No NEW FINAL inference or model scoring is called in this cell.

In [ ]:
# 5 — Inventory and verify both arms' checkpoint SHA + READY-last after completion.
# Read-only until both arms' science checkpoint/VAL receipts are complete.
out = OWNER_PRIVATE / "TTM_HDATA_S1_OWNER_READY_LAST_SCIENCE_READBACK_20261113.json"
if not RUN_ROOT.exists():
    print("HDATA_S1_READY_LAST=NOT_EXECUTED_NO_RUN_ROOT")
else:
    subprocess.run([sys.executable, str(READBACK), "--mode", "inventory",
                    "--root", str(RUN_ROOT)], check=True)
    complete = all((RUN_ROOT / ("ARM_" + arm) / "FINAL_UNADJUDICATED.json").exists()
                   for arm in ("H0", "H1"))
    if not complete:
        print("HDATA_S1_READY_LAST=HOLD_INCOMPLETE_NO_SCIENCE_RESULT")
        print("Never restart from an in-flight step after last READY. Preserve evidence.")
    elif out.exists():
        print("HDATA_S1_READY_LAST_RECEIPT_EXISTS_NO_OVERWRITE=" + sha_file(out))
    else:
        subprocess.run([sys.executable, str(READBACK), "--mode", "verify-complete",
                        "--root", str(RUN_ROOT), "--lock", str(LOCK),
                        "--lock-sha", LOCK_SHA, "--out", str(out)], check=True)
        print("HDATA_S1_NEW_FINAL=SEALED_NO_INFERENCE")
        print("HDATA_S1_UNBLIND=NOT_EXECUTED")
        print("OWNER_READY_LAST_RECEIPT_SHA256=" + sha_file(out))